# Quickstart: Querying PDF With Astra and LangChain

### A question-answering demo using Astra DB and LangChain, powered by Vector Search

#### Pre-requisites:

You need a **_Serverless Cassandra with Vector Search_** database on [Astra DB](https://astra.datastax.com) to run this demo. As outlined in more detail [here](https://docs.datastax.com/en/astra-serverless/docs/vector-search/quickstart.html#_prepare_for_using_your_vector_database), you should get a DB Token with role _Database Administrator_ and copy your Database ID: these connection parameters are needed momentarily.

You also need an [OpenAI API Key](https://cassio.org/start_here/#llm-access) for this demo to work.

#### What you will do:

- Setup: import dependencies, provide secrets, create the LangChain vector store;
- Run a Question-Answering loop retrieving the relevant headlines and having an LLM construct the answer.

Install the required dependencies:

Import the packages you'll need:

In [12]:
# LangChain components to use
from langchain_classic.vectorstores.cassandra import Cassandra
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper
from langchain_openai import OpenAI,OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()
import os

# Support for dataset retrieval with Hugging Face
from datasets import load_dataset
# With CassIO, the engine powering the Astra DB integration in LangChain,
# you will also initialize the DB connection:
# import cassio
from langchain_astradb import AstraDBVectorStore    

In [14]:
from PyPDF2 import PdfReader

### Setup

In [15]:
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN") # enter your Application Token
ASTRA_DB_ID = os.getenv("ASTRA_DB_ID") # enter your Database ID

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") # enter your OpenAI key

#### Provide your secrets:

Replace the following with your Astra DB connection details and your OpenAI API key:

In [16]:
# provide the path of  pdf file/files.
pdfreader = PdfReader('DBMS_Full_Notes.pdf')

In [17]:
from typing_extensions import Concatenate
# read text from pdf
raw_text = ''
for i, page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        raw_text += content

In [18]:
raw_text

'LEC-1: Introduction to DBMS \n1. What is Data?\na. Data is a collection of raw, unorganized facts and details like text, observations, figures, symbols,\nand descriptions of things etc.In other words, data does not carry any specific purpose and has no significance by itself.\nMoreover, data is measured in terms of bits and bytes – which are basic units of information in the\ncontext of computer storage and processing.\nb. Data can be recorded and doesn’t have any meaning unless processed.\n2. Types of Data\na. Quanti tative\ni.Numerical form\nii.Weight, volume, cost of an item.\nb. Qualitative\ni.Descriptive, but not numerical.\nii.Name, gender, hair color of a person.\n3. What is Information?\na. Info. Is processed, organized, and structured data.\nb. It provides context of the data and enables decision making.\nc. Processed data that make sense to us.\nd. Information is extracted from the data, by analyzing and interpreti ng pieces of data.\ne. E.g., you have data of all the people

Initialize the connection to your database:

_(do not worry if you see a few warnings, it's just that the drivers are chatty about negotiating protocol versions with the DB.)_

In [ ]:
# cassio.init(token=ASTRA_DB_APPLICATION_TOKEN, database_id=ASTRA_DB_ID)

ERROR:cassandra.connection:Closing connection <AsyncoreConnection(136828348925792) 56eada22-55b6-4100-aeab-a83bfdf9f82e-us-east1.db.astra.datastax.com:29042:3bfaeefd-3cd1-41fe-a7a3-d89ab7f88095> due to protocol error: Error from server: code=000a [Protocol error] message="Beta version of the protocol used (5/v5-beta), but USE_BETA flag is unset"


Create the LangChain embedding and LLM objects for later usage:

In [19]:
llm = OpenAI(openai_api_key=OPENAI_API_KEY)
embedding = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

Create your LangChain vector store ... backed by Astra DB!

In [21]:
# astra_vector_store = Cassandra(
#     embedding=embedding,
#     table_name="qa_mini_demo",
#     session=None,
#     keyspace=None,
# ).
astra_vector_store = AstraDBVectorStore(
    embedding=embedding,
    collection_name="qa_mini_demo",  # 'table_name' is now 'collection_name'
    api_endpoint="https://76a6c836-7251-40fb-80fa-723e210c2ba6-us-east-2.apps.astra.datastax.com",
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=None,                  # 'keyspace' is now 'namespace'
)

In [22]:
from langchain_text_splitters import CharacterTextSplitter
# We need to split the text using Character Text Split such that it sshould not increse token size
text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 800,
    chunk_overlap  = 200,
    length_function = len,
)
texts = text_splitter.split_text(raw_text)

Created a chunk of size 805, which is longer than the specified 800


In [25]:
texts[:50]

['LEC-1: Introduction to DBMS \n1. What is Data?\na. Data is a collection of raw, unorganized facts and details like text, observations, figures, symbols,\nand descriptions of things etc.In other words, data does not carry any specific purpose and has no significance by itself.\nMoreover, data is measured in terms of bits and bytes – which are basic units of information in the\ncontext of computer storage and processing.\nb. Data can be recorded and doesn’t have any meaning unless processed.\n2. Types of Data\na. Quanti tative\ni.Numerical form\nii.Weight, volume, cost of an item.\nb. Qualitative\ni.Descriptive, but not numerical.\nii.Name, gender, hair color of a person.\n3. What is Information?\na. Info. Is processed, organized, and structured data.',
 'b. Qualitative\ni.Descriptive, but not numerical.\nii.Name, gender, hair color of a person.\n3. What is Information?\na. Info. Is processed, organized, and structured data.\nb. It provides context of the data and enables decision maki

### Load the dataset into the vector store



In [26]:
# Just storing 50 chunks into the dataset
astra_vector_store.add_texts(texts[:50])

print("Inserted %i headlines." % len(texts[:50]))

astra_vector_index = VectorStoreIndexWrapper(vectorstore=astra_vector_store)

Inserted 50 headlines.


### Run the QA cycle

Simply run the cells and ask a question -- or `quit` to stop. (you can also stop execution with the "▪" button on the top toolbar)

Here are some suggested questions:
- _What is database?_
- _What is Foreign Key?_

In [27]:
first_question = True
while True:
    if first_question:
        query_text = input("\nEnter your question (or type 'quit' to exit): ").strip()
    else:
        query_text = input("\nWhat's your next question (or type 'quit' to exit): ").strip()

    if query_text.lower() == "quit":
        break

    if query_text == "":
        continue

    first_question = False

    print("\nQUESTION: \"%s\"" % query_text)
    answer = astra_vector_index.query(query_text, llm=llm).strip()
    print("ANSWER: \"%s\"\n" % answer)

    print("FIRST DOCUMENTS BY RELEVANCE:")
    for doc, score in astra_vector_store.similarity_search_with_score(query_text, k=4):
        print("    [%0.4f] \"%s ...\"" % (score, doc.page_content[:84]))


QUESTION: "What is databse"
ANSWER: "Database is an electronic place/system where data is stored in a way that it can be easily accessed, managed, and updated. It is a collection of interrelated data and a set of programs to access and manage that data. The primary goal of a database is to provide a way to store and retrieve information that is both convenient and efficient."

FIRST DOCUMENTS BY RELEVANCE:
    [0.9231] "pr
esented through words, language, thoughts, and ideas.
g. Data isn’t sufficient fo ..."
    [0.9128] "4. What is RDBMS? (Relational Database Management System)
1. Software that enable us ..."
    [0.9115] "i.Open Database Connectivity ( ODBC), Microsoft “C”.
ii.Java Database Connectivity ( ..."
    [0.9103] "LEC-1: Introduction to DBMS 
1. What is Data?
a. Data is a collection of raw, unorga ..."

QUESTION: "What is Primary Key ?"
ANSWER: "Primary Key is an attribute or set of attributes that can uniquely identify each entity in the entity set. It is used as the prim